In [2]:
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langchain_deepseek import ChatDeepSeek
from rich import print

from dotenv import load_dotenv

load_dotenv(override=True)

#0. LLM モデルに接続
model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)


#1. 状態を定義
class OverAllState(MessagesState):
    username: str
    output: str


#2. ノードを定義
def node_a(state: OverAllState) -> OverAllState:
    return {
        "messages": [HumanMessage("こんにちは、私は" + state["username"])],
    }


def llm_node(state: OverAllState) -> OverAllState:
    res = model.invoke(state["messages"])
    return {
        "messages": [res],
        "output": res.content
    }


#3. グラフを構築
builder = StateGraph(state_schema=OverAllState)

builder.add_node("node_a", node_a)
builder.add_node("llm_node", llm_node)

builder.add_edge(START, "node_a")
builder.add_edge("node_a", "llm_node")
builder.add_edge("llm_node", END)

graph = builder.compile()

#4. グラフを実行
result = graph.invoke({"username": "田中"})
print(result)
print(result["output"])


{
    'messages': [
        HumanMessage(
            content='こんにちは、私は田中',
            additional_kwargs={},
            response_metadata={},
            id='c078733a-b2cb-4e82-a79e-2e88e68f61c9'
        ),
        AIMessage(
            content='こんにちは、田中さん！  \nお会いできてうれしいです。どのようにお手伝いしましょうか？  
\n何か質問や相談があれば、遠慮なくお聞かせください。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 47,
                    'prompt_tokens': 12,
                    'total_tokens': 59,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 12
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '9d75858c-e538-4510-a5b9-cb2189af3f85',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019ffe04-ac3c-7c52-a2f0-aa766f22c49a-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 12,
                'output_tokens': 47,
                'total_tokens': 59,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        )
    ],
    'username': '田中',
    'output': 'こんにちは、田中さん！  \nお会いできてうれしいです。どのようにお手伝いしましょうか？  
\n何か質問や相談があれば、遠慮なくお聞かせください。'
}

こんにちは、田中さん！  
お会いできてうれしいです。どのようにお手伝いしましょうか？  
何か質問や相談があれば、遠慮なくお聞かせください。